# Simulação - Fase 1: Fatiamento e Detecção
Este notebook simula o sistema de fatiamento de mosaico e a inferência via YOLOv8, criado para o projeto de Contagem de Público em Imagens Aéreas de Drone.

In [1]:
!pip install ultralytics opencv-python-headless numpy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\USUARIO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Módulo Tiling (Fatiamento)

In [2]:
import cv2
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple

@dataclass
class Tile:
    image: np.ndarray
    x_offset: int
    y_offset: int

def slice_mosaic(mosaic: np.ndarray, tile_size: int = 1280, overlap: float = 0.2) -> List[Tile]:
    h, w = mosaic.shape[:2]
    step = int(tile_size * (1 - overlap))
    tiles: List[Tile] = []
    for y in range(0, h, step):
        for x in range(0, w, step):
            x_end = min(x + tile_size, w)
            y_end = min(y + tile_size, h)
            x_start = max(0, x_end - tile_size)
            y_start = max(0, y_end - tile_size)
            crop = mosaic[y_start:y_end, x_start:x_end]
            tiles.append(Tile(image=crop, x_offset=x_start, y_offset=y_start))
    return tiles

## Módulo Detector (YOLOv8)

In [3]:
from ultralytics import YOLO

PERSON_CLASSES = {0: "person"}

@dataclass
class Detection:
    x1: float
    y1: float
    x2: float
    y2: float
    confidence: float
    class_name: str = "person"

def _cuda_available() -> bool:
    import torch
    return torch.cuda.is_available()

def run_yolo_on_tile(model: YOLO, tile: Tile, conf_threshold: float = 0.30, imgsz: int = 736, iou_threshold: float = 0.45) -> List[Detection]:
    results = model.predict(
        source=tile.image,
        imgsz=imgsz,
        conf=conf_threshold,
        iou=iou_threshold,
        classes=list(PERSON_CLASSES.keys()),
        verbose=False,
        device="cuda" if _cuda_available() else "cpu",
    )
    detections: List[Detection] = []
    for result in results:
        if result.boxes is None:
            continue
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            if cls_id not in PERSON_CLASSES:
                continue
            detections.append(Detection(
                x1=x1 + tile.x_offset,
                y1=y1 + tile.y_offset,
                x2=x2 + tile.x_offset,
                y2=y2 + tile.y_offset,
                confidence=conf,
            ))
    return detections

## Teste Integrado (Phase 1)

In [4]:
def test_tiling():
    print("Testing Tiling Module...")
    mosaic = np.zeros((2000, 3000, 3), dtype=np.uint8)
    tiles = slice_mosaic(mosaic, tile_size=1280, overlap=0.2)
    print(f"Número de tiles gerados: {len(tiles)}")
    for i, t in enumerate(tiles):
        print(f"Tile {i}: offset=({t.x_offset}, {t.y_offset}), shape={t.image.shape}")
    assert len(tiles) > 0, "Nenhum tile gerado!"
    print("✓ Tiling testado com sucesso!\n")

def test_detector():
    print("Testing Detector Module...")
    model = YOLO("yolov8n.pt")
    img = np.zeros((1280, 1280, 3), dtype=np.uint8)
    dummy_tile = Tile(image=img, x_offset=500, y_offset=500)
    print("Executando YOLOv8 no tile...")
    detections = run_yolo_on_tile(model, dummy_tile, conf_threshold=0.1)
    print(f"Detecções encontradas: {len(detections)}")
    print("✓ Detector testado com sucesso!\n")

test_tiling()
test_detector()

Testing Tiling Module...
Número de tiles gerados: 6
Tile 0: offset=(0, 0), shape=(1280, 1280, 3)
Tile 1: offset=(1024, 0), shape=(1280, 1280, 3)
Tile 2: offset=(1720, 0), shape=(1280, 1280, 3)
Tile 3: offset=(0, 720), shape=(1280, 1280, 3)
Tile 4: offset=(1024, 720), shape=(1280, 1280, 3)
Tile 5: offset=(1720, 720), shape=(1280, 1280, 3)
✓ Tiling testado com sucesso!

Testing Detector Module...
Executando YOLOv8 no tile...
Detecções encontradas: 0
✓ Detector testado com sucesso!

